# NASA Turbofan RUL · Colab Runner

Notebook puente: monta Drive, clona o actualiza el repositorio y ejecuta las etapas definidas en `src/`. **Orquesta, no implementa.**

| Sesión | Runtime | Secciones |
|---|---|---|
| 1 | CPU | 0 → 1 → 2 → 3 |
| 2 | GPU | 0 → 4 → 5 |

Cambiar el tipo de runtime destruye la VM: hay que repetir la sección 0. Los datos y modelos persisten en Drive.

> Las etapas cuyo módulo todavía no está implementado lanzan `NotImplementedError`.

## 0. Preparación (en ambas sesiones)

In [ ]:
# 0.1 Montar Google Drive (antes de importar src)
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# 0.2 Clonar o actualizar el repositorio
import os
import subprocess
import sys

REPO_URL = "https://github.com/facupeletay/nasa-turbofan-rul.git"
BRANCH = "main"
REPO_DIR = "/content/nasa-turbofan-rul"


def run(*args):
    # A diferencia de `!comando`, detiene el notebook si el comando falla
    result = subprocess.run(args, capture_output=True, text=True)
    print((result.stdout + result.stderr).strip())
    result.check_returncode()


if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    run("git", "-C", REPO_DIR, "pull", "--ff-only", "origin", BRANCH)
else:
    run("git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

In [ ]:
# 0.3 Dependencias (Colab ya trae la mayoría; no reinstala lo que está presente)
run(sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt")

In [ ]:
# 0.4 Recarga automática de src/ tras un git pull, y rutas activas
%load_ext autoreload
%autoreload 2

from src import config

config.ensure_directories()
print(config.describe())

In [ ]:
# 0.5 Prueba rápida de métricas
from src.metrics import nasa_asymmetric_score, rmse

assert rmse([100, 50], [110, 40]) == 10.0
assert nasa_asymmetric_score([100], [110]) > nasa_asymmetric_score([100], [90])
print("Métricas OK")

## 1. ETL (CPU)

In [ ]:
from src import etl

SUBSET = config.DEFAULT_SUBSET
data = etl.run_etl(SUBSET)

# Verificación anti-leakage: ningún motor en Train y Val a la vez
assert set(data.train[config.UNIT_COL]).isdisjoint(data.val[config.UNIT_COL])
print(len(data.train), len(data.val), len(data.test), data.feature_cols)

## 2. EDA (CPU)

Pendiente. Preguntas a responder: ¿aporta `s_6`?, ¿cómo se comportan `s_9` y `s_14` por motor?, ¿dónde aparece el codo de degradación respecto de 125 ciclos?

## 3. Baseline XGBoost (CPU)

In [ ]:
from src import baseline_xgb

xgb_metrics = baseline_xgb.run_baseline(config.DEFAULT_SUBSET)
xgb_metrics

## 4. Modelos secuenciales (GPU)

Cambiar a *Entorno de ejecución → Cambiar tipo → GPU* y repetir la sección 0.

In [ ]:
import torch

assert torch.cuda.is_available(), "Este runtime no tiene GPU asignada"
print(torch.cuda.get_device_name(0))

In [ ]:
from src import models_dl

lstm_metrics = models_dl.run_training("lstm", config.DEFAULT_SUBSET)
lstm_metrics

In [ ]:
from src import models_dl

cnn_metrics = models_dl.run_training("cnn1d", config.DEFAULT_SUBSET)
cnn_metrics

## 5. Cierre

Los modelos quedan en `MyDrive/nasa_rul/models/`. Desconectar el runtime de GPU para no consumir unidades de cómputo.

In [ ]:
from google.colab import runtime

# runtime.unassign()  # Descomentar para liberar la GPU